[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch51_code.ipynb)

# 第51章　経過観察と時系列のAI：「変化」を捉える

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 変化検出のパイプラインを、手順で追う

In [ ]:
import numpy as np

warped_prior = register(prior, current, mode="rigid")       # まず剛体。胸腹部など
                                                            # 必要な部位だけ自由度を上げる
diff = normalize(current) - normalize(warped_prior)         # 強度をそろえてから差分
change = np.abs(diff) > CHANGE_TH                           # ノイズの大きさを超えた変化のみ

## 治療効果を、規則で測る：RECISTの自動化

In [ ]:
# 標的はベースラインで決めて固定する。毎回選び直すと総和が不連続に変動し、判定の一貫性が損なわれる。
targets = baseline.targets                      # {lesion_id, 種別(node/non_node), 臓器}
for t in targets:
    t.mask = locate(t.lesion_id, current)       # 同じIDの病変を今回の画像で追跡する
    # locate は「消失」を空のマスクで、「撮れていない・追跡できない」を None で返す前提
    if t.mask is None:
        t.diam, t.state = None, "評価不能"       # 消失（径0mm）とは別の値にする
    elif not t.mask.any():
        t.diam, t.state = 0.0, "消失"
    elif t.kind == "node":
        t.diam = short_axis_mm(t.mask)          # リンパ節は「短径」
    else:
        t.diam = axial_longest_diameter_mm(t.mask)   # 非リンパ節は軸位断の「最長径」
# 評価不能の標的を除いて足すと総和が小さく出て、偽のPR・CRになる。一つでもあれば総和を作らない
sld = None if any(t.diam is None for t in targets) else sum(t.diam for t in targets)   # 径の総和(SLD)
# 標的だけでは総合効果判定にならない。非標的の評価と新病変を必ず併せて渡す。
resp = recist_overall(sld, baseline_sld, nadir_sld,
                      non_target=assess_non_target(current),   # 消失/非CR非PD/明らかな増悪
                      new_lesion=has_new(current),
                      unevaluable=[t.lesion_id for t in targets if t.diam is None])

## 動きそのものを診る：時空間モデル

In [ ]:
import torch

# 各フレームを空間エンコードし、時間方向に統合する2段階の構成（基本構造）
feats = torch.stack([cnn(frame) for frame in clip])   # (T, C, H, W) 各フレームの空間特徴
h = temporal_model(feats)                             # ConvLSTM / 時間アテンション等で統合
pred = head(h)                                        # 例：駆出率、壁運動異常、灌流指標